In [4]:
%load_ext autoreload
%autoreload 2
import os, sys, re
import numpy as np
import pandas as pd
import scanpy as sc
import torch
import time

import matplotlib.pyplot as plt
from matplotlib import cm

os.chdir("/ssd/users/Wergillius/Project/PINN_dynamics")

/ssd/users/Wergillius/miniforge3/envs/torchcfm/lib/python3.10/site-packages/anndata/utils.py:429: FutureWarning: Importing read_csv from `anndata` is deprecated. Import anndata.io.read_csv instead.
  warnings.warn(msg, FutureWarning)
/ssd/users/Wergillius/miniforge3/envs/torchcfm/lib/python3.10/site-packages/anndata/utils.py:429: FutureWarning: Importing read_text from `anndata` is deprecated. Import anndata.io.read_text instead.
  warnings.warn(msg, FutureWarning)
/ssd/users/Wergillius/miniforge3/envs/torchcfm/lib/python3.10/site-packages/anndata/utils.py:429: FutureWarning: Importing read_excel from `anndata` is deprecated. Import anndata.io.read_excel instead.
  warnings.warn(msg, FutureWarning)
/ssd/users/Wergillius/miniforge3/envs/torchcfm/lib/python3.10/site-packages/anndata/utils.py:429: FutureWarning: Importing read_mtx from `anndata` is deprecated. Import anndata.io.read_mtx instead.
  warnings.warn(msg, FutureWarning)
/ssd/users/Wergillius/miniforge3/envs/torchcfm/lib/python3

In [5]:
device = 'cuda:3'

In [6]:
# larry = sc.read_h5ad("data/Weinreb2020_fate_prediction.h5ad")
adata = sc.read_h5ad("data/klein_subset.h5ad")

cellstate_key = "DM_EigenVectors_multiscaled"
timepoint_key = 'timepoint_tx_days'

Fate_bias = pd.read_csv("data/Weinreb/F_obs.csv", index_col=0)
Fate_bias.index = Fate_bias.index.astype(str)

overlapped_cbs = np.intersect1d(Fate_bias.index, adata.obs_names)

Fate_bias = Fate_bias.loc[overlapped_cbs]

In [7]:
X_test = adata[overlapped_cbs].obsm[cellstate_key]

for wasstein distance

# load PRESCIENT model

In [8]:
from prescient.train.model import SimpleNamespace, AutoGenerator

[KeOps] Warning : Cuda libraries were not detected on the system or could not be loaded ; using cpu only mode


In [9]:
config_path = "logs/PRESCIENT/GOBP_cellcycle_DM-softplus_4_64-1e-06/seed_2/config.pt"

train_pt = "logs/PRESCIENT/GOBP_cellcycle_DM-softplus_4_64-1e-06/seed_2/train.best.pt"

In [10]:
config = SimpleNamespace(**torch.load(config_path))
net = AutoGenerator(config)

checkpoint = torch.load(train_pt, map_location=device)
net.load_state_dict(checkpoint['model_state_dict'])

net = net.to(device)
net

AutoGenerator(
  (net): Sequential(
    (linear1): Linear(in_features=5, out_features=64, bias=True)
    (softplus1): Softplus(beta=1.0, threshold=20.0)
    (linear2): Linear(in_features=64, out_features=64, bias=True)
    (softplus2): Softplus(beta=1.0, threshold=20.0)
    (linear3): Linear(in_features=64, out_features=64, bias=True)
    (softplus3): Softplus(beta=1.0, threshold=20.0)
    (linear4): Linear(in_features=64, out_features=64, bias=True)
    (softplus4): Softplus(beta=1.0, threshold=20.0)
    (linear): Linear(in_features=64, out_features=1, bias=False)
  )
)

In [11]:
num_steps =  (config.train_t[-1] - config.start_t) / config.train_dt
num_steps = int(num_steps)
num_steps

20

Simulate the trajectory by hand

In [12]:
X_test = adata[overlapped_cbs].obsm[cellstate_key]
x_i = torch.tensor(X_test).float().to(device)

for _ in range(num_steps):
    z = torch.randn(x_i.shape[0], x_i.shape[1]) * config.train_sd
    z = z.to(device)
    x_i = net._step(x_i, dt = config.train_dt, z = z)

In [ ]:
DM_t = x_i.detach().cpu().numpy()

In [ ]:
np.save("logs/PRESCIENT/GOBP_cellcycle_DM-softplus_4_64-1e-06/seed_2/Diffusion_map_simulated.npy", DM_t)

Simulate the trajectory for wasserstein

In [13]:
test_clones = ['Clone_3869' , 'Clone_2673' , 'Clone_2831']
cbs = [adata.obs.query("`timepoint_tx_days` == @t & `clones` in @test_clones").index.to_list()
            for t in adata.uns['pop']['t']]
W_cb_indexs = [[np.where(adata.obs_names == cb)[0] for cb in cb_t] for cb_t in cbs]

In [14]:
len(cbs)

3

In [19]:
X_test = adata[cbs[1]].obsm[cellstate_key]
x_i = torch.tensor(X_test).float().to(device)

traj = [x_i.detach().cpu().numpy()]
for _ in range(num_steps):
    z = torch.randn(x_i.shape[0], x_i.shape[1]) * config.train_sd
    z = z.to(device)
    x_i = net._step(x_i, dt = config.train_dt, z = z)

    traj.append(x_i.detach().cpu().numpy())

In [20]:
traj = np.stack(traj)
print(traj.shape)
np.save("logs/PRESCIENT/GOBP_cellcycle_DM-softplus_4_64-1e-06/seed_2/wasserstein_simulated_trajectory.npy", traj)

(21, 65, 5)


In [13]:
from torchcfm.optimal_transport import wasserstein

In [28]:
X_true = adata[cbs[2]].obsm[cellstate_key]
print("W2: \t", wasserstein(x_i.cpu(), torch.tensor(X_true), power=2, method='exact'))

W2: 	 2.0795264804315545


# simulate with PCA

In [32]:
config_path = "logs/PRESCIENT/GOBP_cellcycle_PCA-softplus_1_500-1e-06/seed_2/config.pt"

train_pt = "logs/PRESCIENT/GOBP_cellcycle_PCA-softplus_1_500-1e-06/seed_2/train.best.pt"

In [33]:
config = SimpleNamespace(**torch.load(config_path))
net = AutoGenerator(config)

checkpoint = torch.load(train_pt, map_location=device)
net.load_state_dict(checkpoint['model_state_dict'])

net = net.to(device)

/tmp/ipykernel_2312392/2942462341.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  config = SimpleNamespace(**torch.load(config_path))
/tmp/ipykernel_2312392/2942462341.p

In [34]:
X_test = adata[overlapped_cbs].obsm['X_pca']
x_i = torch.tensor(X_test).float().to(device)

for _ in range(num_steps):
    z = torch.randn(x_i.shape[0], x_i.shape[1]) * config.train_sd
    z = z.to(device)
    x_i = net._step(x_i, dt = config.train_dt, z = z)

In [36]:
PCA_t = x_i.detach().cpu().numpy()

In [37]:
np.save("logs/PRESCIENT/GOBP_cellcycle_PCA-softplus_1_500-1e-06/seed_2/Diffusion_map_simulated.npy", PCA_t)